In [1]:
import GPy
from pbbo import Sushi, Noisifier, EPComparisonGP, ThompsonSampling, BayesianOptimization

In [2]:
objective = Sushi()

# Normalize the objective and add noise:
objective.init_normalize_X() # scale bounds between 0 and 1
objective.init_normalize_Y() # scale function maximum and minimum between 0 and 1
noise_level = 0.05
# add normally distributed noise with std 'noise_level'
# objective = Noisifier(objective, 'add', noise_level)

 /Users/changkun/miniconda3/envs/bo/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3432: RuntimeWarning:Mean of empty slice.
 /Users/changkun/miniconda3/envs/bo/lib/python3.9/site-packages/numpy/core/_methods.py:190: RuntimeWarning:invalid value encountered in double_scalars


In [3]:
# Generate a kernel for the GP
kernel = GPy.kern.RBF(
    input_dim=len(objective.bounds),
    ARD=True,
    variance=1.0,
    lengthscale=[0.5] * len(objective.bounds)
)
kernel.variance.constrain_fixed(1.0, warning=False)
kernel.lengthscale.constrain_fixed([0.5] * len(objective.bounds), warning=False)
kernel

rbf.,value,constraints,priors
variance,1.0,+ve fixed,
lengthscale,"(4,)",+ve fixed,


In [4]:
config = {
    "inference": EPComparisonGP,
    "kernel": kernel,
    "acquisition": ThompsonSampling({
        # How many samples are used in stochastic optimization of EV and QEI
        "acq_samples": 5000,
        # How many different initialization locations are randomly sampled for optimizing the acquisition function}
        "acq_opt_restarts": 30,
        # How many posterior samples are used from the posterior, if -1 all are used. Only matters for MCMC.
        "pool": -1,
    }, {
        # Acquisition function optimizer options, see possible options from scipy optimize options
    }),
    # Batch size >2
    "batch_size": 4,  
    # If comparative feedback is used
    "initial_size": 0,
    # How many direct observations are used in initailization
    "initial_size_direct": 0,
    # Noise standard deviation for observations the likelihood uses
    "noise": noise_level,
    # Max number of observation in the BO loo. Number of iterations is maximum number oservations divided by the batch size.
    "max_num_observations": 60,
}

In [6]:
bo = BayesianOptimization(config)
X, yc = bo.bayesian_optimization(objective)

start to run BO loop


 /Users/changkun/dev/changkun.de/human-preference-prior/backendpy/pbbo.py:500: RuntimeWarning:covariance is not positive-semidefinite.


KeyboardInterrupt: 